<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI_Phase24.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from pathlib import Path
import json, hashlib, os, re, ast, subprocess, sys
from datetime import datetime, timezone

ROOT = Path("/content")
MOUNT = Path("/content/drive")
PROJECT = MOUNT / "MyDrive" / "NeuroFHIR_ICHI2027"
P24_DIR = PROJECT / "phase24_outputs"
PRIV = P24_DIR / "_PRIVATE_DO_NOT_SHARE"

def sha256_file(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def find_all(root, name):
    return list(root.rglob(name)) if root.exists() else []

def find_one(root, name):
    m = find_all(root, name)
    return m[0] if m else None

In [4]:
from google.colab import drive
import shutil

if not (MOUNT / "MyDrive").is_dir():
    if MOUNT.exists() and any(MOUNT.iterdir()):
        backup = ROOT / "phase24_premount_backup"
        if backup.exists():
            shutil.rmtree(backup)
        MOUNT.rename(backup)
    MOUNT.mkdir(parents=True, exist_ok=True)
    drive.mount(str(MOUNT))

PROJECT = MOUNT / "MyDrive" / "NeuroFHIR_ICHI2027"
P24_DIR = PROJECT / "phase24_outputs"
PRIV = P24_DIR / "_PRIVATE_DO_NOT_SHARE"
P24_DIR.mkdir(parents=True, exist_ok=True)
PRIV.mkdir(parents=True, exist_ok=True)

phase23 = find_one(PROJECT, "ICHI2027_Phase23_SHAREABLE.json")
assert phase23 is not None, "Phase23 shareable JSON not found on Drive."
p23 = json.loads(phase23.read_text())
assert p23.get("phase") == "23_original_generic_producer_semantics_and_MRI_adapter_contract"
print("Phase23 loaded:", phase23)

Mounted at /content/drive
Phase23 loaded: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase23_outputs/ICHI2027_Phase23_SHAREABLE.json


In [5]:
REPO = ROOT / "ohdsi_tfl_phase24"
if REPO.exists():
    subprocess.run(["rm","-rf",str(REPO)], check=True)

url = "https://github.com/SANGHATI23/ohdsi-fhir-omop-showcase-demo.git"
subprocess.run(["git","clone","-q",url,str(REPO)], check=True)
subprocess.run(["git","-C",str(REPO),"checkout","-q","a4b7265adabf096e84b957de068e24a5ab853425"], check=True)
actual_commit = subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"], text=True).strip()
assert actual_commit == "a4b7265adabf096e84b957de068e24a5ab853425"

sel = p23["selected_candidate_for_private_manual_review"]
target_name = sel["source_file"]
target_idx = int(sel["cell_index"])
matches = list(REPO.rglob(target_name))
assert matches, f"Selected notebook not found: {target_name}"
target_nb_path = matches[0]

In [6]:
import nbformat

nb = nbformat.read(target_nb_path, as_version=4)
assert target_idx < len(nb.cells)
target_src = nb.cells[target_idx].get("source","")
target_sha = hashlib.sha256(target_src.encode()).hexdigest()
assert target_sha == sel["source_sha256"]

(PRIV / "ICHI2027_Phase24_SELECTED_ORIGINAL_PRODUCER_CELL_PRIVATE.py").write_text(target_src)

print("Selected source:", target_name, "cell", target_idx)
print("SHA verified:", target_sha)

Selected source: FHIRy_pyOMOP_TFL_EncounterAuditRepair_v10.ipynb cell 18
SHA verified: d448fb9a2fa378d63cb199223d7142db627b5d6def53d2507d96fab10808262c


/usr/local/lib/python3.13/dist-packages/nbformat/validator.py:434: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  _validate(nbdict, ref, version, version_minor, relax_add_props)


In [7]:
GENERIC_FIELDS = {
    "source_resource_type","source_df_index","target_omop_table",
    "target_omop_record_id","fidelity_status","warning_code","is_mapping_event"
}

DOMAIN_SPECIFIC_PATTERNS = {
    "medication": r"\b(drug|medication|rxnorm|dose|ingredient|route|drug_exposure)\b",
    "encounter": r"\b(encounter|visit_occurrence|visit_detail|admission|discharge)\b",
    "condition": r"\b(condition|diagnosis|condition_occurrence)\b",
    "procedure": r"\b(procedure|procedure_occurrence)\b",
}

tree = ast.parse(target_src)
names = sorted({n.id for n in ast.walk(tree) if isinstance(n, ast.Name)})
attrs = sorted({n.attr for n in ast.walk(tree) if isinstance(n, ast.Attribute)})
strs = [n.value for n in ast.walk(tree) if isinstance(n, ast.Constant) and isinstance(n.value, str)]

domain_hits = {}
lower_src = target_src.lower()
for k, pat in DOMAIN_SPECIFIC_PATTERNS.items():
    domain_hits[k] = sorted(set(re.findall(pat, lower_src, flags=re.I)))

literal_statuses = sorted({
    s for s in strs if s in {
        "AMBIGUOUS","DROPPED","NORMALIZED","PRESERVED",
        "TRACEABILITY_LOSS","UNMAPPED"
    }
})

referenced_generic_fields = sorted({f for f in GENERIC_FIELDS if f in target_src})

# inspect nearby cells for dependency symbols
window = []
for i in range(max(0,target_idx-4), min(len(nb.cells), target_idx+3)):
    if nb.cells[i].get("cell_type") == "code":
        src = nb.cells[i].get("source","")
        window.append({"index": i, "sha256": hashlib.sha256(src.encode()).hexdigest(), "source": src})

dependency_candidates = sorted({
    x for x in names
    if x.isupper() or x.endswith("_STATUSES") or x.endswith("_RULES")
})

In [8]:
# Determine whether the selected producer is portable without inventing semantics.
# Conservative rule: any domain-specific business logic blocks direct MRI execution.
domain_specific_nonempty = {k:v for k,v in domain_hits.items() if v}

portable_static = (
    len(referenced_generic_fields) >= 6
    and "fidelity_status" in target_src
    and "warning_code" in target_src
    and len(domain_specific_nonempty) == 0
)

# We do not override this conservative gate merely because the file title contains "Encounter".
# Only code-level domain terms count above.
semantic_review = {
    "selected_source_file": target_name,
    "selected_cell_index": target_idx,
    "selected_cell_sha256": target_sha,
    "referenced_generic_fields": referenced_generic_fields,
    "literal_statuses": literal_statuses,
    "domain_specific_code_hits": domain_specific_nonempty,
    "dependency_symbol_candidates": dependency_candidates,
    "portable_static_without_new_domain_rules": portable_static,
}

(PRIV / "ICHI2027_Phase24_SELECTED_PRODUCER_SEMANTIC_REVIEW_PRIVATE.json").write_text(
    json.dumps(semantic_review, indent=2)
)
semantic_review

{'selected_source_file': 'FHIRy_pyOMOP_TFL_EncounterAuditRepair_v10.ipynb',
 'selected_cell_index': 18,
 'selected_cell_sha256': 'd448fb9a2fa378d63cb199223d7142db627b5d6def53d2507d96fab10808262c',
 'referenced_generic_fields': ['fidelity_status',
  'is_mapping_event',
  'source_df_index',
  'source_resource_type',
  'target_omop_record_id',
  'target_omop_table',
  'warning_code'],
 'literal_statuses': ['AMBIGUOUS',
  'DROPPED',
  'NORMALIZED',
  'PRESERVED',
  'TRACEABILITY_LOSS',
  'UNMAPPED'],
 'domain_specific_code_hits': {'encounter': ['encounter', 'visit_occurrence']},
 'dependency_symbol_candidates': ['AMBIGUOUS',
  'BASE_TRANSFORMATION_VERSION',
  'COLLAPSED',
  'CORRECTED_DB',
  'DROPPED',
  'ENCOUNTER_AUDIT',
  'ENCOUNTER_DF',
  'NORMALIZED',
  'PRESERVED',
  'SOURCE_DF',
  'TFL_AUDIT',
  'TRACEABILITY_LOSS',
  'TRANSFORMATION_VERSION',
  'UNMAPPED',
  'VARIANTS',
  'VISIT_MAPPING_RULE_ID',
  'VISIT_RULE'],
 'portable_static_without_new_domain_rules': False}

In [9]:
# Locate private MRI adapter candidates created in earlier phases.
# We only inventory; no participant-level data are exported.
candidate_files = []
patterns = [
    "*phase20*",
    "*Phase20*",
    "*phase18*",
    "*Phase18*",
    "*candidate*",
    "*adapter*",
]
seen = set()
for pat in patterns:
    for p in PROJECT.rglob(pat):
        if p.is_file() and p not in seen:
            seen.add(p)
            candidate_files.append(p)

inventory = []
for p in candidate_files:
    try:
        size = p.stat().st_size
    except Exception:
        continue
    inventory.append({
        "basename": p.name,
        "suffix": p.suffix.lower(),
        "bytes": size,
        "sha256": sha256_file(p) if size < 100_000_000 else None,
    })

(PRIV / "ICHI2027_Phase24_PRIVATE_ADAPTER_FILE_INVENTORY.json").write_text(
    json.dumps(inventory, indent=2)
)

1986

In [10]:
# Optional execution gate.
# We do NOT run the old producer on MRI unless it is statically portable.
# If blocked, that itself is a valid result: Arm C cannot be claimed yet.
execution = {
    "attempted": False,
    "completed": False,
    "reason": None,
    "real_mri_rows_processed": 0,
    "real_mri_audit_rows_generated": 0,
}

if not portable_static:
    execution["reason"] = "BLOCKED_BY_DOMAIN_SPECIFIC_OR_INCOMPLETE_SEMANTICS"
else:
    execution["reason"] = "PORTABLE_STATIC_GATE_PASSED_BUT_EXACT_RUNTIME_DEPENDENCIES_NOT_EXECUTED_WITHOUT_SEPARATE_REPLAY_HARNESS"

In [11]:
shareable = {
    "project": "ICHI2027_neuroimaging_fhir_omop_fidelity",
    "phase": "24_selected_original_generic_producer_portability_gate",
    "run_utc": datetime.now(timezone.utc).isoformat(),
    "status": "EXACT_SELECTED_ORIGINAL_PRODUCER_SEMANTIC_PORTABILITY_AUDIT_NO_INVENTED_GENERIC_MRI_RULES",
    "privacy": "Only aggregate/static public-code findings and private file counts; no participant data or code excerpts exported.",
    "phase23_sha256": sha256_file(phase23),
    "pinned_repo_commit_verified": actual_commit == "a4b7265adabf096e84b957de068e24a5ab853425",
    "selected_original_producer": {
        "source_file": target_name,
        "cell_index": target_idx,
        "source_sha256": target_sha,
        "generic_contract_fields_referenced": referenced_generic_fields,
        "literal_statuses": literal_statuses,
        "domain_specific_code_hit_categories": sorted(domain_specific_nonempty.keys()),
        "portable_static_without_new_domain_rules": portable_static,
    },
    "private_adapter_inventory": {
        "candidate_files_found": len(inventory),
        "csv_or_parquet_candidates": sum(1 for x in inventory if x["suffix"] in {".csv",".parquet"}),
        "json_candidates": sum(1 for x in inventory if x["suffix"] == ".json"),
    },
    "execution": execution,
    "gates": {
        "phase23_upstream_verified": True,
        "selected_original_cell_hash_verified": True,
        "selected_original_producer_semantics_reviewed": True,
        "selected_producer_portable_to_MRI_without_new_rules": portable_static,
        "real_MRI_original_generic_C_executed": False,
        "official_MICDM_load_completed": False,
        "exact_ADNI_units_attested": False,
        "DICOM_UID_semantics_attested": False,
        "Athena_mappings_approved": False,
        "true_four_arm_comparison_completed": False,
    },
    "next_stage_gate": (
        "IF_PORTABLE_STATIC_TRUE_BUILD_EXACT_RUNTIME_REPLAY_HARNESS_FOR_SELECTED_ORIGINAL_PRODUCER_ON_COMMON_DOMAIN_ONLY; "
        "IF_FALSE_DO_NOT_FORCE_ARM_C_AND_DOCUMENT_GENERIC_TFL_NONPORTABILITY_TO_MRI_WITHOUT_NEW_RULES"
    )
}

out = P24_DIR / "ICHI2027_Phase24_SHAREABLE.json"
out.write_text(json.dumps(shareable, indent=2))
print(out)
print(json.dumps(shareable["selected_original_producer"], indent=2))

/content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs/ICHI2027_Phase24_SHAREABLE.json
{
  "source_file": "FHIRy_pyOMOP_TFL_EncounterAuditRepair_v10.ipynb",
  "cell_index": 18,
  "source_sha256": "d448fb9a2fa378d63cb199223d7142db627b5d6def53d2507d96fab10808262c",
  "generic_contract_fields_referenced": [
    "fidelity_status",
    "is_mapping_event",
    "source_df_index",
    "source_resource_type",
    "target_omop_record_id",
    "target_omop_table",
    "warning_code"
  ],
  "literal_statuses": [
    "AMBIGUOUS",
    "DROPPED",
    "NORMALIZED",
    "PRESERVED",
    "TRACEABILITY_LOSS",
    "UNMAPPED"
  ],
  "domain_specific_code_hit_categories": [
    "encounter"
  ],
  "portable_static_without_new_domain_rules": false
}
